# 04 Modeling
Train Linear Regression and Random Forest. Save best pipeline.

In [ ]:
import pandas as pd
import numpy as np
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, root_mean_squared_error, r2_score
from sklearn.model_selection import cross_val_score
import joblib

X_train = pd.read_csv('../data/processed/X_train.csv')
X_test  = pd.read_csv('../data/processed/X_test.csv')
y_train = pd.read_csv('../data/processed/y_train.csv').squeeze()
y_test  = pd.read_csv('../data/processed/y_test.csv').squeeze()
print('Data loaded.')

In [ ]:
# ─── Sanity check & cleanup ───────────────────────────────────────────────────
# Guard against string columns surviving from notebook 03
# (e.g. bool→True/False in CSV, or unencodedcategoricals)

def clean_X(df, name):
    # 1. Convert bool-string columns ("True"/"False") to int
    for col in df.columns:
        if df[col].dtype == object:
            unique_vals = df[col].dropna().unique()
            if set(unique_vals).issubset({'True', 'False'}):
                df[col] = df[col].map({'True': 1, 'False': 0})

    # 2. Drop any remaining non-numeric columns and report them
    str_cols = df.select_dtypes(include='object').columns.tolist()
    if str_cols:
        print(f'[{name}] Dropping {len(str_cols)} non-numeric columns: {str_cols}')
        df = df.drop(columns=str_cols)
    else:
        print(f'[{name}] All {df.shape[1]} columns numeric — OK')
    return df

X_train = clean_X(X_train, 'X_train')
X_test  = clean_X(X_test,  'X_test')

In [ ]:
def evaluate(name, model, X_tr, y_tr, X_te, y_te):
    model.fit(X_tr, y_tr)
    pred = model.predict(X_te)
    mae  = mean_absolute_error(y_te, pred)
    rmse = root_mean_squared_error(y_te, pred)
    r2   = r2_score(y_te, pred)
    cv   = cross_val_score(model, X_tr, y_tr, cv=5, scoring='r2').mean()
    print(f'{name:20s} MAE={mae:.1f}  RMSE={rmse:.1f}  R2={r2:.3f}  CV-R2={cv:.3f}')
    return model, pred

In [ ]:
lr, pred_lr = evaluate('Linear Regression',
    LinearRegression(), X_train, y_train, X_test, y_test)

In [ ]:
rf, pred_rf = evaluate('Random Forest',
    RandomForestRegressor(n_estimators=100, random_state=42),
    X_train, y_train, X_test, y_test)

In [ ]:
# Save best model (Random Forest expected to win)
joblib.dump(rf, '../models/rent_pipeline.pkl')
print('Model saved to models/rent_pipeline.pkl')